# Rapport de performance

Ce rapport lit `data/06_performance/performance_metrics.csv`. Il ne réestime pas les poids et il ne rouvre pas les cours. Le tableau de bord est `reports/performance/dashboard.html`. L'analyse est `notebooks/07_analyse_performance.ipynb`.

Le rendement, la volatilité, le ratio de Sharpe et le drawdown maximal sont ceux de la phase 5. Le calcul s'arrête s'il ne les retrouve pas. Le ratio de Sortino et l'écart de suivi par rapport au fonds iShares sont ajoutés ici.

Un ratio de Sortino vide signifie qu'aucun mois n'est passé sous le SARON. Un ratio d'information vide signifie que l'écart de suivi est nul.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.performance.build_performance import METRICS_OUTPUT

LABELS = {
    "equal_weight": "Équipondéré",
    "minimum_volatility": "Volatilité minimale",
    "maximum_sharpe": "Sharpe maximal",
}


def french(value, digits):
    return f"{float(value):.{digits}f}".replace(".", ",")


def percent(value):
    return french(float(value) * 100, 1) + " %"


def ratio(value):
    if pd.isna(value):
        return "non publié"
    return french(value, 2)


metrics = pd.read_csv(METRICS_OUTPUT)
if set(metrics["strategy"]) != set(LABELS):
    raise RuntimeError("The performance file does not cover the three strategies.")
start = metrics["window_start"].iloc[0]
end = metrics["window_end"].iloc[0]
months = int(metrics["months"].iloc[0])
print(f"Période {start} – {end}, {months} mois.")
print(
    "Ces mesures décrivent le chemin des poids fixés chaque mois avec les soixante mois précédents. "
    "Chaque décision n'utilise que les rendements déjà observés."
)
print(f"La référence de l'écart de suivi est {metrics['reference'].iloc[0]}.")
print()
for strategy, label in LABELS.items():
    row = metrics.loc[metrics["strategy"].eq(strategy)].iloc[0]
    print(label)
    print(
        f"Rendement annualisé {percent(row['annualized_return'])}. "
        f"Volatilité {percent(row['annualized_volatility'])}."
    )
    print(f"Sharpe {ratio(row['sharpe_ratio'])}. Sortino {ratio(row['sortino_ratio'])}.")
    print(f"Drawdown maximal {percent(row['maximum_drawdown'])}.")
    print(f"Mois sous le SARON : {int(row['months_below_risk_free'])}.")
    print(
        f"Écart de suivi {percent(row['tracking_error'])}. "
        f"Ratio d'information {ratio(row['information_ratio'])}."
    )
    print()
print(
    "L'écart de suivi compare chaque portefeuille au rendement en francs "
    "de la valeur liquidative publiée par iShares."
)


Période 2016-10 – 2026-09, 120 mois.
Ces mesures décrivent le chemin des poids fixés chaque mois avec les soixante mois précédents. Chaque décision n'utilise que les rendements déjà observés.
La référence de l'écart de suivi est iShares MSCI Global Sustainable Development Goals ETF.

Équipondéré
Rendement annualisé 8,4 %. Volatilité 12,4 %.
Sharpe 0,72. Sortino 1,14.
Drawdown maximal -17,0 %.
Mois sous le SARON : 47.
Écart de suivi 5,9 %. Ratio d'information 0,33.

Volatilité minimale
Rendement annualisé 6,8 %. Volatilité 11,1 %.
Sharpe 0,66. Sortino 1,03.
Drawdown maximal -19,2 %.
Mois sous le SARON : 45.
Écart de suivi 7,2 %. Ratio d'information 0,05.

Sharpe maximal
Rendement annualisé 10,6 %. Volatilité 12,9 %.
Sharpe 0,86. Sortino 1,38.
Drawdown maximal -20,2 %.
Mois sous le SARON : 44.
Écart de suivi 7,9 %. Ratio d'information 0,52.

L'écart de suivi compare chaque portefeuille au rendement en francs de la valeur liquidative publiée par iShares.
